# TP CNN — détecteur (Transfer Learning)



**Nom :** …  

**Date :** 2026-05-25  

**Cours :** MSc AIB — Méthodes actuelles de l'apprentissage profond



## Énoncé (résumé)

Objectif : se familiariser avec l’usage de CNN pour la **détection** et le **transfer learning** (éviter de réapprendre from scratch), en manipulant des **données image ou vidéo**.



Travail attendu :

- Partir d’un **détecteur pré-entraîné** (COCO / PascalVOC / etc.) et/ou d’un classifieur.

- Choisir un **nouveau cas d’usage** qui vous intéresse (ex : humain/véhicules en CCTV, plaques, espèces de plantes…).

- Trouver un **dataset de détection** adapté à ce cas d’usage.

- **Adapter les classes de sortie** au nouveau cas (transfer learning) et **ré-entraîner** (au minimum les dernières couches).

- **Évaluer** la détection : mAP à différents IoU, AP par classe, precision, recall.

- Décider si un fine-tuning plus profond est nécessaire (dé-geler davantage de couches).



Conseil : familles simples à utiliser : **YOLO**, **Inception**, **RetinaNet** (mais libre choix).



## Livrables (dans ce notebook)

- Choix du cas d’usage + justification (2–5 phrases)

- Description du dataset (source, licence si disponible, nb images/vidéos, nb classes)

- Détails du modèle pré-entraîné et des classes conservées/changées

- Stratégie transfer learning (freeze/unfreeze, hyperparamètres)

- Résultats : mAP50, mAP50-95, AP par classe, precision/recall + interprétation

- Analyse : erreurs typiques, limites, améliorations possibles


In [1]:
# Imports (utilisés dans les cellules suivantes)

import random

from pathlib import Path

from typing import List


## Dataset — provenance et métadonnées

- Source: (indiquer la provenance du dataset utilisé — ex: Roboflow / VIRAT / lien zip / local)
- Licence: (précisez la licence si connue)
- Nombre d'images (train/val): (remplir après inspection)
- Classes: 
  - 
, 
(à renseigner)


In [82]:
# Sauvegarde synthétique des chemins et métriques de run
from pathlib import Path

summary_path = PROJECT_DIR / 'run_summary.txt'
info_lines = []

# Meilleur checkpoint si disponible
try:
    w = weights if 'weights' in globals() else None
    info_lines.append(f'weights: {w}')
except Exception:
    info_lines.append('weights: None')

# Métriques si présentes
try:
    m = metrics
    info_lines.append(f"mAP50-95: {float(getattr(m.box, 'map', float('nan')))}")
    info_lines.append(f"mAP50   : {float(getattr(m.box, 'map50', float('nan')))}")
    info_lines.append(f"Precision: {float(getattr(m.box, 'mp', float('nan')))}")
    info_lines.append(f"Recall   : {float(getattr(m.box, 'mr', float('nan')))}")
except Exception:
    info_lines.append('metrics: unavailable')

# Écrire le résumé
try:
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    with open(summary_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(map(str, info_lines)) + '\n')
    print('Run summary écrit dans:', summary_path)
    print('\n'.join(info_lines))
except Exception as e:
    print('Échec écriture summary:', e)

Run summary écrit dans: C:\r_yolo\run_summary.txt
weights: C:\r_yolo\cctv_person_car_yolov8_finetune_quick\weights\best.pt
mAP50-95: 0.285514818048973
mAP50   : 0.48460953779534827
Precision: 0.5316336972854192
Recall   : 0.5069544797687862


In [84]:
# Tracer et sauvegarder PR-curves / AP par IoU si disponibles
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

out_dir = PROJECT_DIR / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)

def try_save_pr_curves(metrics_obj, out_dir: Path):
    """Tente d'extraire et sauvegarder PR curves depuis l'objet metrics retourné par Ultralytics."""
    try:
        curves = getattr(metrics_obj.box, 'pr', None) or getattr(metrics_obj.box, 'curves', None) or getattr(metrics_obj.box, 'pr_curves', None)
        if not curves:
            print('PR curves non disponibles dans metrics.')
            return
        if isinstance(curves, dict):
            items = curves.items()
        else:
            items = enumerate(curves)
        for cls_id, data in items:
            xs = data.get('recall') if isinstance(data, dict) else None
            ys = data.get('precision') if isinstance(data, dict) else None
            if xs is None or ys is None:
                continue
            plt.figure()
            plt.plot(xs, ys)
            plt.xlabel('recall')
            plt.ylabel('precision')
            name = f'pr_curve_cls_{cls_id}.png'
            plt.savefig(out_dir / name)
            plt.close()
            print('Saved', name)
    except Exception as e:
        print('Erreur sauvegarde PR curves:', e)

if 'metrics' in globals():
    try_save_pr_curves(metrics, out_dir)
else:
    print("Objet 'metrics' indisponible — exécutez d'abord la cellule d'évaluation pour remplir 'metrics'.")

PR curves non disponibles dans metrics.


## 0) Configuration du TP (ultra simple)



Il y'a **qu’un seul truc** à choisir : le *preset* du cas d’usage.



Ensuite il faut exécuter les cellules dans l’ordre.



- TP très simple : preset **CCTV (person/car)**.

- plaques : preset **license_plate** (1 seule classe).

- plantes : preset **plants_3** (3 classes exemple, à renommer si besoin).


In [ ]:
# ---- A CHOISIR (1 seule ligne) ----
# Mets l'une de ces valeurs: "cctv_person_car" | "license_plate" | "plants_3" | "custom"
USE_CASE_PRESET = "cctv_person_car"
# ---------------------------------


# Nom libre pour tes runs
USE_CASE_NAME = USE_CASE_PRESET


# Presets de classes (possible de modifier, mais ce n'est pas obligatoire)
PRESET_CLASSES = {
    "cctv_person_car": ["person", "car"],
    "license_plate": ["license_plate"],
    "plants_3": ["plant_a", "plant_b", "plant_c"],
}


# Si l'option "custom" est choisie, écris tes classes ici
CUSTOM_CLASSES: List[str] = ["class_0"]

if USE_CASE_PRESET == "custom":
    CLASSES: List[str] = CUSTOM_CLASSES
else:
    if USE_CASE_PRESET not in PRESET_CLASSES:
        raise ValueError(f"Preset inconnu: {USE_CASE_PRESET}. Choisis dans {list(PRESET_CLASSES.keys())} ou 'custom'.")
    CLASSES = PRESET_CLASSES[USE_CASE_PRESET]


# Chemin vers ton dataset au format YOLO (voir section 1)
# Laisser comme ça et mettre ton dataset dans: data/<preset>/...
DATASET_DIR = Path("data") / USE_CASE_PRESET


# Mode "test rapide" (optionnel): True => utilise le mini dataset Ultralytics (coco8) juste pour valider le code
# ATTENTION: ça ne remplace pas ton dataset final du devoir.
USE_COCO8_QUICKTEST = False


# Poids pré-entraînés (COCO) : yolov8n.pt (rapide), yolov8s.pt, yolov8m.pt...
PRETRAINED_WEIGHTS = "yolov8n.pt"


# Hyperparams d'entraînement (à ajuster)
IMG_SIZE = 640
BATCH = 4
EPOCHS_FROZEN = 10
# En mode 'rapide' pour test: réduit le nombre d'epochs de finetune
EPOCHS_FINETUNE = 20
# Indique qu'on est en mode quick (utile pour scripts/tests)
QUICK_MODE = True


# Transfer learning : nombre de couches “gelées” (Ultralytics).
FREEZE_LAYERS = 10


# Sorties/expériences
RUN_NAME = f"{USE_CASE_NAME}_yolov8"
# Répertoire de sortie raccourci pour éviter chemins Windows trop longs
PROJECT_DIR = Path("C:/r_yolo")

print("USE_CASE_PRESET =", USE_CASE_PRESET)
print("CLASSES         =", CLASSES)
print("DATASET_DIR     =", DATASET_DIR.resolve())
print("PROJECT_DIR     =", PROJECT_DIR.resolve())
print("QUICKTEST coco8 =", USE_COCO8_QUICKTEST)


USE_CASE_PRESET = cctv_person_car
CLASSES         = ['person', 'car']
DATASET_DIR     = C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car
PROJECT_DIR     = C:\r_yolo
QUICKTEST coco8 = False


In [ ]:
# Remplacer le contenu de data/cctv_person_car/ par celui de car-+-person-1/ (sauf dataset.yaml)
import shutil
from pathlib import Path

src = Path('car-+-person-1')
dst = DATASET_DIR

# Conserver dataset.yaml s'il existe
dataset_yaml = dst / 'dataset.yaml'
tmp_yaml = None
if dataset_yaml.exists():
    tmp_yaml = dst.parent / 'tmp_dataset.yaml'
    shutil.copy2(dataset_yaml, tmp_yaml)

# On supprime tout sauf dataset.yaml
for item in dst.iterdir():
    if item.name != 'dataset.yaml':
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()

# On copie le contenu de src dans dst
for item in src.iterdir():
    target = dst / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# On restaure dataset.yaml si besoin
if tmp_yaml and tmp_yaml.exists():
    shutil.move(tmp_yaml, dataset_yaml)
    print('dataset.yaml conservé.')
print('Contenu de data/cctv_person_car remplacé par car-+-person-1 (sauf dataset.yaml).')

dataset.yaml conservé.
Contenu de data/cctv_person_car remplacé par car-+-person-1 (sauf dataset.yaml).


In [ ]:
# Déplacement automatique des images dans la structure YOLO attendue (version robuste)
import shutil
from pathlib import Path

# Vérifier que la structure est YOLO complète (images/labels train/val)
for sub in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    (DATASET_DIR / sub).mkdir(parents=True, exist_ok=True)

# Dossiers sources possibles
src_train = DATASET_DIR / 'train' / 'images'
src_val = DATASET_DIR / 'valid' / 'images'
src_test = DATASET_DIR / 'test' / 'images'
dst_train = DATASET_DIR / 'images' / 'train'
dst_val = DATASET_DIR / 'images' / 'val'


def move_images(src, dst, label):
    """Déplace tous les fichiers de src vers dst en créant dst si nécessaire.
    Ignore les erreurs individuelles et rapporte le nombre de fichiers déplacés.
    """
    dst.mkdir(parents=True, exist_ok=True)
    moved = 0
    if src.exists() and any(src.iterdir()):
        for img in src.glob('*'):
            try:
                # Utilise shutil.move qui gère cross-device; si échoue, on copiera+supprimera
                shutil.move(str(img), str(dst))
                moved += 1
            except Exception:
                try:
                    target = dst / img.name
                    shutil.copy2(str(img), str(target))
                    img.unlink()
                    moved += 1
                except Exception:
                    print(f"Échec pour: {img}")
        try:
            # essaie de supprimer le dossier source s'il est vide
            src.rmdir()
        except Exception:
            pass
        print(f'Images {label} déplacées: {moved}')
    else:
        print(f'Aucune image à déplacer depuis {src}')


move_images(src_train, dst_train, 'train')
move_images(src_val, dst_val, 'val')
move_images(src_test, dst_val, 'test (vers val)')

print('Réorganisation des images terminée.')


Images train déplacées: 1386
Images val déplacées: 396
Images test (vers val) déplacées: 198
Réorganisation des images terminée.


In [ ]:
# Copier automatiquement les fichiers .txt d'annotations vers labels/train et labels/val
import shutil
from pathlib import Path

dst_train = DATASET_DIR / 'labels' / 'train'
dst_val = DATASET_DIR / 'labels' / 'val'
dst_train.mkdir(parents=True, exist_ok=True)
dst_val.mkdir(parents=True, exist_ok=True)

# Trouver tous les .txt sous DATASET_DIR en excluant labels/train et labels/val
candidates = [p for p in DATASET_DIR.rglob('*.txt') if not (dst_train in p.parents or dst_val in p.parents)]
print(f"Candidates labels trouvés: {len(candidates)}")

moved = 0
for lf in candidates:
    # ignorer non-annotation txt (heuristique: nom 'dataset' ou 'data' -> skip)
    if lf.name.lower().startswith('dataset') or lf.name.lower().startswith('data'):
        continue

    base = lf.stem
    found = None
    # recherche d'image correspondante dans images/train/val
    for ext in ('.jpg', '.jpeg', '.png', '.bmp', '.webp'):
        if (DATASET_DIR / 'images' / 'train' / (base + ext)).exists():
            found = 'train'
            break
        if (DATASET_DIR / 'images' / 'val' / (base + ext)).exists():
            found = 'val'
            break
    if found is None:
        # heuristiques supplémentaires via chemin
        sp = str(lf).lower()
        if '\\train\\' in sp or '/train/' in sp:
            found = 'train'
        elif '\\valid\\' in sp or '/valid/' in sp or '\\val\\' in sp or '/val/' in sp:
            found = 'val'
        elif '\\test\\' in sp or '/test/' in sp:
            found = 'val'
        else:
            found = 'val'  # fallback

    target = dst_train / lf.name if found == 'train' else dst_val / lf.name
    try:
        shutil.copy2(str(lf), str(target))
        moved += 1
    except Exception as e:
        print(f"Échec copie {lf} -> {target}: {e}")

print(f"Annotations copiées: {moved} vers labels/train/labels/val")


Candidates labels trouvés: 1982
Annotations copiées: 1982 vers labels/train/labels/val


## 1) Préparation des données (format YOLO)



Ce notebook suppose un dataset de détection au **format YOLO** :



- `DATASET_DIR/images/train/*.jpg|png`

- `DATASET_DIR/images/val/*.jpg|png`

- `DATASET_DIR/labels/train/*.txt`

- `DATASET_DIR/labels/val/*.txt`



Chaque fichier label YOLO contient des lignes :

`class_id x_center y_center width height` (coordonnées normalisées dans [0,1]).



Si votre dataset est en COCO/PascalVOC : convertissez-le vers YOLO (Roboflow, scripts, etc.).



Dans la cellule suivante, on génère un `dataset.yaml` compatible Ultralytics et on fait des checks simples.


### Télécharger / installer le dataset (si lien .zip disponible)



Le dataset doit se trouver **sur le disque** (dans ce projet) pour entraîner le modèle.



Cas fréquents :

- **Lien direct `.zip`** : utilise la cellule suivante (télécharge + extrait).

- **Kaggle** : prérequis -> un compte + une clé `kaggle.json`.

- **Roboflow** : possible d'exporter en **YOLOv8** (souvent le plus simple), puis on obtient un zip.



But final : le dataset doit se retrouver dans `DATASET_DIR` avec la structure :

- `images/train`, `images/val`

- `labels/train`, `labels/val`


### Télécharger depuis Roboflow (API) — optionnel



Roboflow propose aussi un téléchargement via Python.



Important : **ne mets pas ta clé API en dur dans le notebook**. Utilise une variable d’environnement.


In [ ]:
# Téléchargement Roboflow via API (OPTIONNEL)
#
# Si ton dataset est DÉJÀ téléchargé (zip via navigateur).
# Sans clé, la cellule ne fait rien et ne bloque pas.

import os
import sys
from pathlib import Path
import shutil

USE_ROBOFLOW_API_DOWNLOAD = False  # mets True seulement si télécharger via API


def _sync_tree(src_dir: Path, dst_dir: Path) -> None:
    """Copie le contenu de src_dir vers dst_dir (merge), sans créer de sous-dossier en plus."""
    src_dir = src_dir.resolve()
    dst_dir = dst_dir.resolve()
    dst_dir.mkdir(parents=True, exist_ok=True)

    for item in src_dir.iterdir():
        target = dst_dir / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)


if not USE_ROBOFLOW_API_DOWNLOAD:
    print("Téléchargement Roboflow via API désactivé. Passe à la cellule suivante.")

else:
    api_key = os.environ.get("ROBOFLOW_API_KEY")
    if not api_key:
        print("ROBOFLOW_API_KEY absente -> skip. Passe à la cellule suivante.")
    else:
        # Installation robuste (utilise le Python du kernel)
        !{sys.executable} -m pip install -q --upgrade pip
        !{sys.executable} -m pip install -q roboflow

        from roboflow import Roboflow

        # ---- A RENSEIGNER (copie depuis Roboflow) ----
        ROBOFLOW_WORKSPACE = "test-0wdss"  # slug workspace
        ROBOFLOW_PROJECT = "car-person-nebip"  # slug projet
        ROBOFLOW_VERSION = 1
        # --------------------------------------------

        rf = Roboflow(api_key=api_key)
        project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
        version = project.version(ROBOFLOW_VERSION)

        # Le format "yolov8" correspond au export Ultralytics YOLOv8
        dataset = version.download("yolov8")

        print("Téléchargement Roboflow terminé.")
        print("Dataset téléchargé dans:", dataset.location)
        print("DATASET_DIR attendu par ce notebook:", DATASET_DIR.resolve())

        # On synchronise automatiquement le contenu dans DATASET_DIR pour que les cellules suivantes fonctionnent.
        src = Path(dataset.location)
        dst = DATASET_DIR
        if src.resolve() != dst.resolve():
            print("Copie du dataset vers DATASET_DIR...")
            _sync_tree(src, dst)
            print("Copie terminée.")

        print("Suite: exécute la cellule suivante (Génère dataset.yaml + vérifications basiques).")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to car-+-person-1 in yolov8:: 100%|██████████| 3972/3972 [00:12<00:00, 316.45it/s]


Téléchargement Roboflow terminé.
Dataset téléchargé dans: c:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\car-+-person-1
DATASET_DIR attendu par ce notebook: C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car
Si besoin, déplace/recopie le contenu de dataset.location vers DATASET_DIR.


In [29]:
# Synchronisation du dataset téléchargé Roboflow vers DATASET_DIR (à exécuter UNE FOIS si besoin)
from pathlib import Path
import shutil

src = Path("car-+-person-1")
dst = Path("data") / "cctv_person_car"
dst.mkdir(parents=True, exist_ok=True)

for item in src.iterdir():
    target = dst / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

print("Dataset copié dans", dst.resolve())

Dataset copié dans C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car


In [2]:
# Téléchargement + extraction d'un zip (optionnel)

# Mets l'URL du zip si disponible. Sinon laisse None.



from __future__ import annotations



import zipfile

from urllib.request import urlretrieve



DATASET_ZIP_URL = None  # ex: "https://.../dataset.zip"

ZIP_PATH = Path("dataset.zip")



if DATASET_ZIP_URL is None:

    print("Renseigner DATASET_ZIP_URL si un lien direct .zip est disponible.")

else:

    print("Téléchargement...", DATASET_ZIP_URL)

    urlretrieve(DATASET_ZIP_URL, ZIP_PATH)

    print("Zip téléchargé:", ZIP_PATH.resolve())



    # On extrait dans DATASET_DIR (défini en cellule 5)

    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    print("Extraction dans:", DATASET_DIR.resolve())

    with zipfile.ZipFile(ZIP_PATH, "r") as z:

        z.extractall(DATASET_DIR)



    # Nettoyage optionnel

    # ZIP_PATH.unlink(missing_ok=True)



    print("Extraction terminée.")

    print("Astuce: si le zip contient un sous-dossier (ex: dataset/...), déplace son contenu dans DATASET_DIR.")


Renseigner DATASET_ZIP_URL si un lien direct .zip est disponible.


In [ ]:
# Génère dataset.yaml + vérifications basiques
#
# Astuce Roboflow: si l'exportiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii en YOLOv8, le zip contient souvent un fichier `data.yaml`.
# Dans ce cas, on l'utilise directement pour éviter les soucis d'ordre de classes.


def _count_files(folder: Path, exts: tuple[str, ...]) -> int:
    if not folder.exists():
        return 0
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts)


def _unique_resolved(paths: list[Path]) -> list[Path]:
    unique: list[Path] = []
    seen = set()
    for p in paths:
        rp = p.resolve()
        if rp not in seen:
            unique.append(rp)
            seen.add(rp)
    return unique


def _find_existing_dataset_yaml(dataset_dir: Path) -> Path | None:
    """Trouve un YAML existant et privilégie `data.yaml` (Roboflow) sur `dataset.yaml`.

    Recherche dans:
    - dataset_dir
    - dataset_dir/*
    - le dossier du notebook (.)
    - ./*

    Pourquoi: Roboflow télécharge souvent dans un dossier à la racine (ex: ./my-dataset-1/data.yaml).
    """

    dataset_dir = dataset_dir.resolve()
    roots = [dataset_dir, *list(dataset_dir.glob("*")), Path(".").resolve(), *list(Path(".").resolve().glob("*"))]

    data_candidates: list[Path] = []
    dataset_candidates: list[Path] = []

    for root in roots:
        if not root.exists() or not root.is_dir():
            continue
        data_candidates += list(root.glob("data.yaml"))
        dataset_candidates += list(root.glob("dataset.yaml"))

    data_unique = _unique_resolved(data_candidates)
    dataset_unique = _unique_resolved(dataset_candidates)

    if len(data_unique) == 1:
        return data_unique[0]
    if len(data_unique) > 1:
        print("⚠️ Plusieurs 'data.yaml' trouvés (Roboflow). Choisis-en un manuellement :")
        for p in data_unique:
            print("-", p)
        return None

    if len(dataset_unique) == 1:
        return dataset_unique[0]
    if len(dataset_unique) > 1:
        print("⚠️ Plusieurs 'dataset.yaml' trouvés. Choisis-en un manuellement :")
        for p in dataset_unique:
            print("-", p)

    return None


def write_ultralytics_dataset_yaml(dataset_dir: Path, classes: List[str]) -> Path:
    dataset_dir = dataset_dir.resolve()
    yaml_path = dataset_dir / "dataset.yaml"

    content = [
        f"path: {dataset_dir.as_posix()}",
        "train: images/train",
        "val: images/val",
        f"nc: {len(classes)}",
        "names:",
    ]
    content += [f"  {i}: {name}" for i, name in enumerate(classes)]

    yaml_path.write_text("\n".join(content) + "\n", encoding="utf-8")
    return yaml_path


def sanity_check_yolo_dataset(dataset_dir: Path, num_classes: int, max_checks: int = 200) -> None:
    img_train = dataset_dir / "images" / "train"
    img_val = dataset_dir / "images" / "val"
    lbl_train = dataset_dir / "labels" / "train"
    lbl_val = dataset_dir / "labels" / "val"

    n_train_img = _count_files(img_train, (".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    n_val_img = _count_files(img_val, (".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    n_train_lbl = _count_files(lbl_train, (".txt",))
    n_val_lbl = _count_files(lbl_val, (".txt",))

    print("Images train:", n_train_img)
    print("Images val  :", n_val_img)
    print("Labels train:", n_train_lbl)
    print("Labels val  :", n_val_lbl)

    if n_train_img == 0 or n_val_img == 0:
        print("⚠️ Dataset incomplet: images train/val manquantes.")
        print("   Structure attendue:")
        print("   - images/train, images/val")
        print("   - labels/train, labels/val")

    label_files: list[Path] = []
    if lbl_train.exists():
        label_files += list(lbl_train.rglob("*.txt"))
    if lbl_val.exists():
        label_files += list(lbl_val.rglob("*.txt"))

    checked = 0
    bad_lines = 0
    for lf in label_files:
        if checked >= max_checks:
            break
        checked += 1

        text = lf.read_text(encoding="utf-8", errors="ignore").strip()
        if not text:
            continue

        for line in text.splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                bad_lines += 1
                continue

            class_id = int(float(parts[0]))
            if class_id < 0 or class_id >= num_classes:
                bad_lines += 1
                continue

            coords = list(map(float, parts[1:]))
            if any(c < 0.0 or c > 1.0 for c in coords):
                bad_lines += 1

    print(f"Labels checkés: {checked}, lignes suspectes: {bad_lines}")


if USE_COCO8_QUICKTEST:
    print("Mode quicktest activé: on utilisera 'coco8.yaml' (mini dataset Ultralytics).")
    data_yaml = "coco8.yaml"

else:
    DATASET_DIR.mkdir(parents=True, exist_ok=True)

    existing_yaml = _find_existing_dataset_yaml(DATASET_DIR)
    if existing_yaml is not None:
        print("YAML existant détecté (ex: Roboflow):", existing_yaml)
        data_yaml = str(existing_yaml)

    else:
        yaml_path = write_ultralytics_dataset_yaml(DATASET_DIR, CLASSES)
        print("YAML écrit:", yaml_path)
        data_yaml = str(yaml_path.resolve())

    # Sanity check seulement si la structure est celle attendue (images/labels)
    # (un export Roboflow peut avoir train/valid au lieu de images/train, labels/train)
    if (DATASET_DIR / "images").exists() and (DATASET_DIR / "labels").exists():
        sanity_check_yolo_dataset(DATASET_DIR, num_classes=len(CLASSES))

    else:
        print("Note: structure non-standard (probablement Roboflow). On se base sur le YAML existant.")


⚠️ Plusieurs 'data.yaml' trouvés (Roboflow). Choisis-en un manuellement :
- C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car\data.yaml
- C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\car-+-person-1\data.yaml
YAML écrit: C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car\dataset.yaml
Images train: 1386
Images val  : 594
Labels train: 0
Labels val  : 0
Labels checkés: 0, lignes suspectes: 0


In [35]:
# Force le YAML à celui du dossier DATASET_DIR pour éviter toute ambiguïté
data_yaml = str((DATASET_DIR / "dataset.yaml").resolve())
print("YAML utilisé pour l'entraînement :", data_yaml)

YAML utilisé pour l'entraînement : C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car\dataset.yaml


## 2) Modèle détecteur (YOLOv8 pré-entraîné)



Ici on utilise **YOLOv8** pré-entraîné sur **COCO** (transfer learning).



Étapes :

1. Installer `ultralytics`

2. Charger `PRETRAINED_WEIGHTS`

3. Faire une prédiction “baseline” sur une image/une frame pour vérifier que tout fonctionne


In [12]:
# Installation (à exécuter une seule fois)

# Si vous avez déjà un environnement configuré, vous pouvez commenter cette cellule.

import sys



python_exe = sys.executable

!{python_exe} -m pip install -q "ultralytics>=8.2" opencv-python matplotlib


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# Chargement du modèle + test baseline

from __future__ import annotations

from pathlib import Path
import random

from ultralytics import YOLO


def _list_images(folder: Path) -> list[Path]:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    if not folder.exists():
        return []
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts]


def _try_find_one_image_from_data_yaml(data_yaml_path: str) -> Path | None:
    """Trouve une image candidate à partir du dataset YAML.

    Roboflow/Ultralytics met souvent les images dans:
    - train/images, valid/images, test/images
    ou
    - images/train, images/val

    On essaie plusieurs heuristiques pour éviter de demander un chemin à la main.
    """

    yml = Path(data_yaml_path)
    if not yml.exists():
        return None

    # 1) Dossiers standards autour du YAML
    base = yml.parent
    candidates_dirs = [
        base / "images" / "val",
        base / "images" / "valid",
        base / "images" / "train",
        base / "valid" / "images",
        base / "val" / "images",
        base / "train" / "images",
        base / "test" / "images",
    ]

    for d in candidates_dirs:
        imgs = _list_images(d)
        if imgs:
            return random.choice(imgs)

    # 2) Fallback: chercher une image n'importe où (1 niveau) autour du YAML
    for d in [base, *[p for p in base.glob("*") if p.is_dir()]]:
        imgs = _list_images(d)
        if imgs:
            return random.choice(imgs)

    return None


model = YOLO(PRETRAINED_WEIGHTS)
print("Modèle chargé:", PRETRAINED_WEIGHTS)

# Option: testez sur une image locale (mettez votre chemin)
# Exemple: TEST_IMAGE = Path("chemin/vers/une_image.jpg")
TEST_IMAGE = None

if TEST_IMAGE is None:
    # Si la cellule YAML est executée(celle qui définit data_yaml), on tente auto.
    try:
        TEST_IMAGE = _try_find_one_image_from_data_yaml(str(data_yaml))  # type: ignore[name-defined]
    except Exception:
        TEST_IMAGE = None

if TEST_IMAGE is not None:
    print("Image choisie pour le test:", Path(TEST_IMAGE).resolve())
    _ = model.predict(source=str(TEST_IMAGE), imgsz=IMG_SIZE, conf=0.25, save=True)
    print("Prédiction baseline OK. Résultats sauvegardés dans runs/...")
else:
    print("Aucune image trouvée automatiquement.")
    print("Option 1: renseigne TEST_IMAGE manuellement (chemin vers un .jpg/.png).")
    print("Option 2: saute ce test baseline et passe directement à l'entraînement.")


Modèle chargé: yolov8n.pt
Image choisie pour le test: C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\car-+-person-1\valid\images\d1b6485a23c4a11b_jpg.rf.a539e363b4fd2b8aa84aa9c0c50d70b3.jpg

image 1/1 C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Mthodes actuelles de l'apprentissage profond\tps\car-+-person-1\valid\images\d1b6485a23c4a11b_jpg.rf.a539e363b4fd2b8aa84aa9c0c50d70b3.jpg: 640x640 1 person, 1 car, 409.8ms
Speed: 16.9ms preprocess, 409.8ms inference, 18.0ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Mthodes actuelles de l'apprentissage profond\tps\runs\detect\predict
Prédiction baseline OK. Résultats sauvegardés dans runs/...


## 3) Entraînement (Transfer Learning)



Stratégie recommandée en TP :

1. **Phase A (frozen)** : on gèle une partie du backbone (`FREEZE_LAYERS`) et on ré-entraîne surtout la tête de détection.

2. **Phase B (fine-tuning)** : on dégèle davantage (voire tout) et on continue à faible LR pour s’adapter au domaine.



À discuter dans le rapport :

- Pourquoi ce choix de freeze/unfreeze ?

- Quels hyperparamètres (epochs, batch, imgsz) et pourquoi ?


In [71]:
# Entraînement phase A: freeze (transfer learning "léger")

# Remarque: la première exécution va télécharger les weights si nécessaire.



model = YOLO(PRETRAINED_WEIGHTS)



results_frozen = model.train(

    data=data_yaml,

    imgsz=IMG_SIZE,

    epochs=EPOCHS_FROZEN,

    batch=BATCH,

    freeze=FREEZE_LAYERS,

    project=str(PROJECT_DIR),

    name=f"{RUN_NAME}_frozen",

)



print("Run frozen terminé.")


New https://pypi.org/project/ultralytics/8.4.54 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.9 torch-2.12.0+cpu CPU (11th Gen Intel Core i7-11370H @ 3.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Mthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_widt

In [73]:
# Entraînement phase B: fine-tuning (on dégèle plus)
# On repart du meilleur checkpoint de la phase A.

from pathlib import Path
import os

# Préférence: utilise le checkpoint explicite qui a été généré par le quick run
_EXPLICIT_BEST = Path(r"C:/r_yolo/cctv_person_car_yolov8_frozen/weights/best.pt")

def _find_best_frozen_weights() -> Path | None:
    """Recherche le best.pt de la phase A en privilégiant le chemin explicite."""
    # 1) checkpoint explicite (court) — évite problèmes de chemins Windows trop longs
    if _EXPLICIT_BEST.exists():
        return _EXPLICIT_BEST
    # 2) emplacement attendu sous PROJECT_DIR/<run>_frozen/weights/best.pt
    cand_fr = (PROJECT_DIR / f"{RUN_NAME}_frozen" / "weights" / "best.pt").resolve()
    if cand_fr.exists():
        return cand_fr
    # 3) recherche récursive pour un run contenant 'frozen' dans son nom
    for p in PROJECT_DIR.rglob("best.pt"):
        try:
            if 'frozen' in str(p.parent.parent):
                return p
        except Exception:
            continue
    # 4) fallback: premier best.pt trouvé sous PROJECT_DIR
    for p in PROJECT_DIR.rglob("best.pt"):
        return p
    return None

best_weights = _find_best_frozen_weights()

if not best_weights or not best_weights.exists():
    print("⚠️ best.pt introuvable pour la phase A (frozen). Vérifie que la phase A s'est bien terminée.")
    print("Aide: cherche manuellement dans runs/ si besoin.")
else:
    print(f"best.pt utilisé pour fine-tuning: {best_weights}")
    # Chargement du modèle depuis le checkpoint frozen
    from ultralytics import YOLO
    model_ft = YOLO(str(best_weights))

    # Quick finetune pour validation (réduire si on veut tester rapidement)
    QUICK_FINETUNE = False
    if QUICK_FINETUNE:
        epochs_to_run = min(2, max(1, int(os.environ.get('QUICK_FINETUNE_EPOCHS', 2))))
    else:
        epochs_to_run = EPOCHS_FINETUNE

    results_ft = model_ft.train(
        data=data_yaml,
        imgsz=IMG_SIZE,
        epochs=epochs_to_run,
        batch=BATCH,
        freeze=0,
        lr0=1e-4,
        project=str(PROJECT_DIR),
        name=f"{RUN_NAME}_finetune",
    )
    print("Run finetune terminé.")


best.pt utilisé pour fine-tuning: C:\r_yolo\cctv_person_car_yolov8_frozen\weights\best.pt
New https://pypi.org/project/ultralytics/8.4.54 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.53  Python-3.13.9 torch-2.12.0+cpu CPU (11th Gen Intel Core i7-11370H @ 3.30GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Mthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car\dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.01

## 4) Évaluation (mAP, AP par classe, precision/recall)



On évalue au minimum :

- **mAP@0.5** (souvent appelé mAP50)

- **mAP@0.5:0.95** (mAP50-95)

- AP par classe, precision, recall



Comparez :

- modèle pré-entraîné (optionnel)

- après phase A (frozen)

- après phase B (fine-tuning)


In [74]:
# Validation / métriques
# Par défaut, on évalue le modèle fine-tuné si disponible, sinon le modèle frozen.

from pathlib import Path

def _pick_trained_weights() -> Path | None:
    """Recherche best.pt en privilégiant le checkpoint finetune explicite, puis chemins usuels."""
    # 1) checkpoint finetune explicite court (évite problèmes Windows path length)
    _EXPLICIT_FINETUNE = Path(r"C:/r_yolo/cctv_person_car_yolov8_finetune_quick/weights/best.pt")
    if _EXPLICIT_FINETUNE.exists():
        return _EXPLICIT_FINETUNE
    # 2) emplacement attendu sous PROJECT_DIR/<run>_finetune/weights/best.pt
    cand_ft = (PROJECT_DIR / f"{RUN_NAME}_finetune" / "weights" / "best.pt").resolve()
    if cand_ft.exists():
        return cand_ft
    # 3) emplacement frozen
    cand_fr = (PROJECT_DIR / f"{RUN_NAME}_frozen" / "weights" / "best.pt").resolve()
    if cand_fr.exists():
        return cand_fr
    # 4) fallback: premier best.pt trouvé sous PROJECT_DIR
    for p in PROJECT_DIR.rglob("best.pt"):
        return p
    return None

weights = _pick_trained_weights()

if weights is None:
    print("⚠️ Aucun poids entraîné trouvé. Lance l'entraînement d'abord.")
else:
    print(f"Poids utilisés pour l'évaluation: {weights}")
    eval_model = YOLO(str(weights))
    metrics = eval_model.val(data=data_yaml, imgsz=IMG_SIZE, batch=BATCH)
    print("Weights:", weights)
    try:
        print("mAP50-95:", float(metrics.box.map))
        print("mAP50   :", float(metrics.box.map50))
        print("mAP75   :", float(metrics.box.map75))
        print("Precision:", float(metrics.box.mp))
        print("Recall   :", float(metrics.box.mr))
    except Exception:
        print(metrics)


Poids utilisés pour l'évaluation: C:\r_yolo\cctv_person_car_yolov8_finetune_quick\weights\best.pt
Ultralytics 8.4.53  Python-3.13.9 torch-2.12.0+cpu CPU (11th Gen Intel Core i7-11370H @ 3.30GHz)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 216.786.2 MB/s, size: 64.2 KB)
val: Scanning C:\Users\riqwi\Desktop\rick\academik\master AI aivancity\cours\certificat3\MSc AIB - Méthodes actuelles de l'apprentissage profond\tps\data\cctv_person_car\labels\val.cache... 594 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 594/594 108.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 149/149 1.7it/s 1:250.6sss
                   all        594       1095      0.532      0.507      0.485      0.286
                person        310        576      0.627      0.698      0.689      0.434
                   car        284        519      0.437      0.

In [75]:
# AP par classe (si dispo)

if weights is not None:

    try:

        names = eval_model.names  # dict id -> name

        ap5095 = getattr(metrics.box, "ap", None)

        ap50 = getattr(metrics.box, "ap50", None)



        print("\nAP par classe:")

        for class_id, class_name in names.items():

            ap5095_i = float(ap5095[class_id]) if ap5095 is not None else None

            ap50_i = float(ap50[class_id]) if ap50 is not None else None

            print(f"- {class_id:>2} {class_name:<20}  AP50-95={ap5095_i}  AP50={ap50_i}")

    except Exception as e:

        print("AP par classe non disponible via cette version Ultralytics:", repr(e))



AP par classe:
-  0 person                AP50-95=0.4335883998169339  AP50=0.688518269543993
-  1 car                   AP50-95=0.13744123628101207  AP50=0.28070080604670355


In [85]:
# Courbe ROC (Receiver Operating Characteristic) — seuil de confiance
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

if 'metrics' not in globals() or weights is None:
    print("Métriques ou poids indisponibles. ROC non calculée.")
else:
    try:
        # Récupère les courbes de confiance vs FPR/TPR si disponibles dans metrics
        # Ultralytics stocke parfois les résultats sous F1, confusion matrix, etc.
        
        # Approche 1: tenter d'accéder directement aux données ROC dans metrics
        has_roc = hasattr(metrics.box, 'roc') or hasattr(metrics.box, 'roc_curve')
        
        if has_roc:
            # Si ROC est disponible, on l'utilise
            roc_data = getattr(metrics.box, 'roc', None) or getattr(metrics.box, 'roc_curve', None)
            if roc_data:
                plt.figure(figsize=(8, 6))
                plt.plot(roc_data[0], roc_data[1], lw=2, label=f'ROC')
                plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
                plt.xlabel('False Positive Rate (1 - Specificity)')
                plt.ylabel('True Positive Rate (Sensitivity)')
                plt.title('ROC Curve (Detection)')
                plt.legend(loc='lower right')
                plt.grid(alpha=0.3)
                out_dir = PROJECT_DIR / 'figures'
                out_dir.mkdir(parents=True, exist_ok=True)
                plt.savefig(out_dir / 'roc_curve.png', dpi=150, bbox_inches='tight')
                plt.close()
                print("Courbe ROC sauvegardée :", out_dir / 'roc_curve.png')
        else:
            # Approche 2: Dérivation manuelle basée sur les seuils de confiance
            # (moins précis, mais pédagogique)
            print("ROC directe non disponible. Génération simplifiée basée sur les métriques.")
            print("Conseil: une ROC complète nécessiterait les scores bruts de confiance de chaque détection.")
            
            # Simulation d'une ROC basique (pédagogique)
            # On varie un seuil hypothétique et simule TPR/FPR
            thresholds = np.linspace(0, 1, 50)
            tpr_list = []
            fpr_list = []
            
            # Récupère precision et recall actuels (à seuil par défaut)
            precision = float(getattr(metrics.box, 'mp', 0.5))
            recall = float(getattr(metrics.box, 'mr', 0.5))
            
            # Simulation: on simule que recall diminue avec le seuil et precision augmente
            for th in thresholds:
                # Heuristique simple: plus le seuil monte, plus recall baisse, plus precision monte
                simulated_recall = max(0, recall * (1 - th * 0.5))
                simulated_precision = min(1, precision + th * 0.3)
                
                # TPR = recall (pour la détection)
                tpr = simulated_recall
                # FPR est approximativement (1 - precision) * (1 - th)
                fpr = max(0, (1 - simulated_precision) * (1 - th * 0.5))
                
                tpr_list.append(tpr)
                fpr_list.append(fpr)
            
            plt.figure(figsize=(8, 6))
            plt.plot(fpr_list, tpr_list, 'b-', lw=2, label=f'ROC Curve (simulated)')
            plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('ROC Curve — Detection (Confidence threshold variation)')
            plt.legend(loc='lower right')
            plt.grid(alpha=0.3)
            out_dir = PROJECT_DIR / 'figures'
            out_dir.mkdir(parents=True, exist_ok=True)
            plt.savefig(out_dir / 'roc_curve_simulated.png', dpi=150, bbox_inches='tight')
            plt.close()
            print("Courbe ROC simulée sauvegardée :", out_dir / 'roc_curve_simulated.png')
            print("Note: pour une ROC précise, il faudrait les scores bruts de toutes les détections.")
    except Exception as e:
        print("Erreur génération ROC:", e)


ROC directe non disponible. Génération simplifiée basée sur les métriques.
Conseil: une ROC complète nécessiterait les scores bruts de confiance de chaque détection.
Courbe ROC simulée sauvegardée : C:\r_yolo\figures\roc_curve_simulated.png
Note: pour une ROC précise, il faudrait les scores bruts de toutes les détections.


In [76]:
# Inference sur images / vidéo (optionnel, recommandé)

# Remplacez SOURCE par un dossier, une image, ou une vidéo.



SOURCE = None  # ex: DATASET_DIR / "images" / "val"  ou  "video.mp4"



weights = _pick_trained_weights()

if SOURCE is None:

    print("Renseignez SOURCE pour lancer l'inférence.")

elif weights is None:

    print("⚠️ Aucun poids entraîné trouvé.")

else:

    infer_model = YOLO(str(weights))

    _ = infer_model.predict(

        source=str(SOURCE),

        imgsz=IMG_SIZE,

        conf=0.25,

        save=True,

        project=str(PROJECT_DIR),

        name=f"{RUN_NAME}_predict",

    )

    print("Inférence terminée. Voir runs/... pour les sorties.")


Renseignez SOURCE pour lancer l'inférence.


## 5) Analyse / Discussion (obligatoire)



Répondez brièvement (bullet points acceptés) :



- **Qualité du dataset** : assez d’images ? annotations correctes ? domaine proche/loin de COCO ?

- **Erreurs typiques** : faux positifs ? objets manqués ? objets petits/occlus ? confusion entre classes ?

- **Effet du transfer learning** : frozen vs fine-tuning : qu’est-ce qui s’améliore/dégrade ?

- **Choix techniques** : pourquoi ces hyperparamètres et ce niveau de freeze ?

- **Pistes d’amélioration** : plus de données, augmentation, résolution, re-labeling, entraînement plus long, etc.



Checklist métriques à inclure :

- mAP50, mAP50-95

- AP par classe

- precision / recall


## Annexes (optionnel) — Vidéo / extraction de frames



Si votre dataset est vidéo (ex. CCTV), une approche simple est :

1. Extraire des frames (1 frame toutes les N frames)

2. Annoter ces images (LabelImg, CVAT, Roboflow…)

3. Entraîner en détection sur les images annotées


In [77]:
# Extraction simple de frames depuis une vidéo (optionnel)

# Utilisez ceci pour créer un sous-ensemble d'images à annoter.



import cv2



VIDEO_PATH = None  # ex: "cctv.mp4"

FRAMES_DIR = Path("frames")

EVERY_N_FRAMES = 30

MAX_FRAMES = 500



if VIDEO_PATH is None:

    print("Renseignez VIDEO_PATH pour extraire des frames.")

else:

    FRAMES_DIR.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(VIDEO_PATH))

    if not cap.isOpened():

        raise RuntimeError(f"Impossible d'ouvrir la vidéo: {VIDEO_PATH}")



    frame_idx = 0

    saved = 0

    while saved < MAX_FRAMES:

        ok, frame = cap.read()

        if not ok:

            break

        if frame_idx % EVERY_N_FRAMES == 0:

            out = FRAMES_DIR / f"frame_{saved:06d}.jpg"

            cv2.imwrite(str(out), frame)

            saved += 1

        frame_idx += 1



    cap.release()

    print(f"Frames sauvegardées: {saved} dans {FRAMES_DIR.resolve()}")


Renseignez VIDEO_PATH pour extraire des frames.
